In [11]:
%%writefile convtea.py
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import re
import csv
import json
import time
import math
import shutil
import random
import argparse
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, List, ContextManager
from collections import Counter
from contextlib import nullcontext

import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.datasets import ImageFolder

import timm
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)

# =============================================================================
# CONVNEXTV2-T ONLY
# =============================================================================

BACKBONE_KEY = "convnextv2_t"
TIMM_ID = "convnextv2_tiny.fcmae_ft_in1k"
DEFAULT_FEAT_DIM = 768


def _probe_feat_dim() -> int:
    """
    CPU-only probe — no CUDA pages created.
    empty_cache() after deletion prevents allocator fragmentation at init.
    """
    try:
        m = timm.create_model(
            TIMM_ID,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )
        m.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            out = m(dummy)
        feat_dim = int(out.shape[-1])
        del out, dummy, m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return feat_dim
    except Exception as e:
        warnings.warn(f"Could not probe feat_dim: {e}. Using default {DEFAULT_FEAT_DIM}.")
        return DEFAULT_FEAT_DIM


# =============================================================================
# UTILITIES
# =============================================================================

def set_seed(seed: int, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.backends.cuda.matmul.allow_tf32 = False
            torch.backends.cudnn.allow_tf32 = False
        except Exception:
            pass
        try:
            torch.use_deterministic_algorithms(True)
        except Exception as e:
            warnings.warn(f"Could not enable deterministic algorithms: {e}")
    else:
        # P100: TF32 is not available (Volta/Pascal arch, not Ampere),
        # but benchmark=True still helps cuDNN pick fast kernels.
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        except Exception:
            pass


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_autocast_ctx(use_amp: bool) -> ContextManager:
    # P100 supports FP16 AMP (introduced on Pascal).
    enabled = bool(use_amp and torch.cuda.is_available())
    if not enabled:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return torch.cuda.amp.autocast(enabled=True)


def make_grad_scaler(use_amp: bool):
    enabled = bool(use_amp and torch.cuda.is_available())
    try:
        return torch.cuda.amp.GradScaler(enabled=enabled)
    except Exception:
        if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
            return torch.amp.GradScaler(enabled=enabled)
        raise


def rgb_to_hsv_torch(rgb01: torch.Tensor) -> torch.Tensor:
    r, g, b = rgb01[:, 0:1], rgb01[:, 1:2], rgb01[:, 2:3]
    maxc, _ = rgb01.max(dim=1, keepdim=True)
    minc, _ = rgb01.min(dim=1, keepdim=True)
    v = maxc
    delta = maxc - minc
    eps = 1e-10

    s = delta / torch.clamp(maxc, min=eps)
    s = torch.where(maxc > eps, s, torch.zeros_like(s))

    delta_safe = torch.clamp(delta, min=eps)
    h_r = (g - b) / delta_safe
    h_g = (b - r) / delta_safe + 2.0
    h_b = (r - g) / delta_safe + 4.0

    idx = rgb01.argmax(dim=1, keepdim=True)
    mask = delta > eps

    h = torch.zeros_like(delta, dtype=rgb01.dtype)
    h = torch.where((idx == 0) & mask, h_r, h)
    h = torch.where((idx == 1) & mask, h_g, h)
    h = torch.where((idx == 2) & mask, h_b, h)
    h = torch.remainder(h / 6.0, 1.0)

    return torch.clamp(torch.cat([h, s, v], dim=1), 0.0, 1.0)


def hsv_to_sincos_sv(hsv01: torch.Tensor) -> torch.Tensor:
    h, s, v = hsv01[:, 0:1], hsv01[:, 1:2], hsv01[:, 2:3]
    ang = 2.0 * math.pi * h
    return torch.cat([torch.sin(ang), torch.cos(ang), s, v], dim=1)


def normalize_hsv_rep(x: torch.Tensor) -> torch.Tensor:
    hsin, hcos = x[:, 0:1], x[:, 1:2]
    s = (x[:, 2:3] - 0.5) / 0.25
    v = (x[:, 3:4] - 0.5) / 0.25
    return torch.cat([hsin, hcos, s, v], dim=1)


def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


def _json_safe_scalar(x):
    if x is None:
        return None
    if isinstance(x, (int, float, str, bool)):
        return x
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        val = float(x)
        return None if (np.isnan(val) or np.isinf(val)) else val
    if isinstance(x, torch.Tensor):
        return x.item() if x.ndim == 0 else x.detach().cpu().numpy().tolist()
    if isinstance(x, np.ndarray):
        return x.item() if x.ndim == 0 else x.tolist()
    return str(x)


def strip_module_prefix(sd):
    return {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}


def add_module_prefix(sd):
    return {(k if k.startswith("module.") else f"module.{k}"): v for k, v in sd.items()}


def load_state_dict_robust(model: nn.Module, state_dict: dict) -> None:
    for fn in [lambda x: x, strip_module_prefix, add_module_prefix]:
        try:
            model.load_state_dict(fn(state_dict), strict=True)
            return
        except Exception:
            pass
    for fn in [lambda x: x, strip_module_prefix, add_module_prefix]:
        try:
            result = model.load_state_dict(fn(state_dict), strict=False)
            if result.missing_keys or result.unexpected_keys:
                warnings.warn(
                    f"Loaded with strict=False | Missing: {result.missing_keys} | "
                    f"Unexpected: {result.unexpected_keys}"
                )
            return
        except Exception:
            pass
    raise RuntimeError("Could not load state dict with any strategy.")


# =============================================================================
# EMA
# P100 note: single GPU, 16 GB.  Shadow lives on GPU for fast in-place updates.
# No DataParallel GPU-0 bottleneck, so keeping shadow on GPU is safe.
# Shadow is serialised to CPU on checkpoint save to keep .pth files small.
# =============================================================================

class EMA:
    """
    Exponential Moving Average of model parameters.
    Shadow tensors live on the same device as the model (GPU) for fast
    in-place updates.  They are moved to CPU only when writing checkpoints.
    """

    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.model = model
        self.decay = decay
        self.shadow: Dict[str, torch.Tensor] = {}
        self.backup: Dict[str, torch.Tensor] = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.data.detach().clone()  # same device as param

    def update(self):
        """In-place GPU update — no host/device transfers."""
        for name, p in self.model.named_parameters():
            if p.requires_grad:
                if name not in self.shadow:
                    self.shadow[name] = p.data.detach().clone()
                else:
                    self.shadow[name].mul_(self.decay).add_(p.data, alpha=(1.0 - self.decay))

    def apply_shadow(self):
        """Swap model params with shadow for evaluation."""
        self.backup = {}
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.shadow:
                self.backup[name] = p.data.detach().clone()
                p.data.copy_(self.shadow[name])

    def restore(self):
        """Restore original params after EMA evaluation."""
        if not self.backup:
            return
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.backup:
                p.data.copy_(self.backup[name])
        self.backup = {}


# =============================================================================
# CONFIG
# P100-specific defaults:
#   batch_size=64          single GPU, no DataParallel scatter bottleneck
#   gradient_accumulation_steps=1   effective batch = 64, one clean step
#   num_workers=4          P100 Kaggle kernels have 4 CPU cores
#   use_data_parallel=False  single GPU; DP adds overhead with 1 device
#   deterministic=False    benchmark=True -> faster cuDNN kernels
# =============================================================================

@dataclass
class Config:
    timm_id: str = TIMM_ID
    feat_dim: int = DEFAULT_FEAT_DIM
    pretrained: bool = True
    drop_path_rate: float = 0.2
    num_classes: int = 7

    data_root: str = "/kaggle/input/datasets/saifullahsharafatfb/tea-leaf701515/tea_leaf_processed_dataset/tea_leaf_processed_dataset"
    input_size: int = 224

    # P100: single 16 GB GPU — batch_size=64 safe without DataParallel.
    # T4x2 users: set batch_size=32, gradient_accumulation_steps=2.
    batch_size: int = 64
    num_workers: int = 4

    img_mean: Tuple[float, float, float] = (0.485, 0.456, 0.406)
    img_std:  Tuple[float, float, float] = (0.229, 0.224, 0.225)

    use_hsv_branch: bool = False
    hsv_use_sincos: bool = True
    gate_warmup_epochs: int = 5
    hsv_embed_dim: int = 128
    hsv_dropout: float = 0.1
    fuse_dropout: float = 0.2
    gate_hidden: int = 256
    gate_vector: bool = False

    epochs: int = 80
    warmup_epochs: int = 5
    lr: float = 5e-4
    warmup_lr_init: float = 1e-6
    weight_decay: float = 0.05
    min_lr: float = 1e-6
    label_smoothing: float = 0.1
    grad_clip_norm: float = 1.0
    gradient_accumulation_steps: int = 1   # P100: no accumulation needed
    use_amp: bool = True                   # P100 supports FP16 (Pascal arch)
    use_ema: bool = False
    ema_decay: float = 0.9998
    ema_eval_every: int = 5                # run EMA val every N epochs
    use_class_weights: bool = False
    early_stopping_patience: int = 25

    color_jitter: float = 0.2
    hue_jitter: float = 0.1
    use_gaussian_blur: bool = False
    gaussian_blur_prob: float = 0.1
    rrc_scale_min: float = 0.85

    run_dir: str = "/kaggle/working/experiments_ablation"
    experiment_name: str = "convnextv2t_run"
    log_interval: int = 100
    save_cm_png: bool = True
    ckpt_temp_dir: str = "/kaggle/working/temp"
    save_epoch_checkpoints: bool = False
    save_epoch_every: int = 5
    keep_last_n_checkpoints: int = 3
    compute_val_auc: bool = False

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    deterministic: bool = False       # False -> benchmark=True (faster on P100)
    use_data_parallel: bool = False   # P100 is single GPU; DP = pure overhead

    def resolve_backbone(self):
        self.feat_dim = _probe_feat_dim()
        print("\n🔍 Backbone resolved:")
        print(f"   key:      {BACKBONE_KEY}")
        print(f"   timm_id:  {self.timm_id}")
        print(f"   feat_dim: {self.feat_dim}")
        print("   notes:    ConvNeXt-V2-Tiny only")

    def validate(self):
        if not Path(self.data_root).exists():
            raise FileNotFoundError(f"data_root not found: {self.data_root}")
        if self.batch_size < 1:
            raise ValueError("batch_size must be >= 1")
        if self.epochs < 1:
            raise ValueError("epochs must be >= 1")
        if not 0 <= self.label_smoothing < 1:
            raise ValueError("label_smoothing must be in [0,1)")
        if self.gradient_accumulation_steps < 1:
            raise ValueError("gradient_accumulation_steps must be >= 1")
        if self.lr <= 0:
            raise ValueError("lr must be > 0")
        if not 0.0 <= self.ema_decay <= 1.0:
            raise ValueError("ema_decay must be in [0,1]")
        if self.ema_eval_every < 1:
            raise ValueError("ema_eval_every must be >= 1")
        if self.batch_size < 32:
            warnings.warn(
                f"batch_size={self.batch_size} is unusually small for a P100. "
                "Consider batch_size=64 (single GPU, 16 GB)."
            )


# =============================================================================
# MODEL
# =============================================================================

class RGBHSVModel(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.use_hsv_branch = cfg.use_hsv_branch
        self.hsv_use_sincos = cfg.hsv_use_sincos
        self.gate_vector = cfg.gate_vector
        self.feat_dim = cfg.feat_dim

        self.register_buffer("img_mean", torch.tensor(cfg.img_mean).view(1, 3, 1, 1))
        self.register_buffer("img_std",  torch.tensor(cfg.img_std).view(1, 3, 1, 1))

        self.backbone = timm.create_model(
            cfg.timm_id,
            pretrained=cfg.pretrained,
            num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
            global_pool="avg",
        )

        if self.use_hsv_branch:
            in_ch = 4 if self.hsv_use_sincos else 3

            self.hsv_branch = nn.Sequential(
                nn.Conv2d(in_ch, 16, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(16),
                nn.GELU(),
                nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.GELU(),
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(p=cfg.hsv_dropout),
                nn.Linear(32, cfg.hsv_embed_dim),
                nn.GELU(),
            )

            self.hsv_proj = nn.Sequential(
                nn.Dropout(p=cfg.fuse_dropout),
                nn.Linear(cfg.hsv_embed_dim, self.feat_dim),
            )

            gate_out_dim = self.feat_dim if self.gate_vector else 1
            self.gate_mlp = nn.Sequential(
                nn.Linear(self.feat_dim + cfg.hsv_embed_dim, cfg.gate_hidden),
                nn.GELU(),
                nn.Dropout(p=cfg.fuse_dropout),
                nn.Linear(cfg.gate_hidden, gate_out_dim),
            )
            nn.init.constant_(self.gate_mlp[-1].bias, -2.0)

            self.classifier = nn.Sequential(
                nn.Dropout(p=cfg.fuse_dropout),
                nn.Linear(self.feat_dim, self.feat_dim // 2),
                nn.GELU(),
                nn.Dropout(p=cfg.fuse_dropout),
                nn.Linear(self.feat_dim // 2, cfg.num_classes),
            )
        else:
            self.classifier = nn.Linear(self.feat_dim, cfg.num_classes)

    def forward(self, x_norm: torch.Tensor, return_gate: bool = False, gate_alpha: float = 1.0):
        feat = self.backbone(x_norm)

        if not self.use_hsv_branch:
            logits = self.classifier(feat)
            if return_gate:
                gate = torch.zeros((logits.size(0), 1), device=logits.device, dtype=logits.dtype)
                return logits, gate
            return logits

        autocast_ctx = torch.cuda.amp.autocast(enabled=False) if torch.cuda.is_available() else nullcontext()
        with autocast_ctx:
            x_rgb01 = (x_norm.float() * self.img_std.float() + self.img_mean.float()).clamp(0.0, 1.0)
            x_hsv = rgb_to_hsv_torch(x_rgb01)
            if self.hsv_use_sincos:
                x_hsv_rep = normalize_hsv_rep(hsv_to_sincos_sv(x_hsv))
            else:
                x_hsv_rep = x_hsv

        x_hsv_rep = x_hsv_rep.to(dtype=feat.dtype)
        hsv_emb = self.hsv_branch(x_hsv_rep)

        gate_in = torch.cat([feat.float(), hsv_emb.float()], dim=1)
        gate = torch.sigmoid(self.gate_mlp(gate_in))

        if not self.gate_vector:
            gate_for_fuse = gate.expand(-1, self.feat_dim)
            gate_stat = gate
        else:
            gate_for_fuse = gate
            gate_stat = gate.mean(dim=1, keepdim=True)

        hsv_feat = self.hsv_proj(hsv_emb)
        gate_for_fuse = gate_for_fuse.to(dtype=feat.dtype)
        hsv_feat = hsv_feat.to(dtype=feat.dtype)

        fused = feat + float(gate_alpha) * gate_for_fuse * hsv_feat
        logits = self.classifier(fused)

        if return_gate:
            return logits, gate_stat
        return logits


# =============================================================================
# CHECKPOINT MANAGER
# =============================================================================

class CheckpointManager:
    def __init__(self, temp_root: Path, final_root: Path, experiment_name: str):
        self.temp_dir = temp_root / experiment_name
        self.final_dir = final_root / experiment_name
        self.pid = os.getpid()

        self.temp_dir.mkdir(parents=True, exist_ok=True)
        for sub in ["model", "config", "logs", "metrics", "raw_outputs", "figures"]:
            (self.final_dir / sub).mkdir(parents=True, exist_ok=True)

        print(f"\n📁 Directories:")
        print(f"   Temp:  {self.temp_dir}")
        print(f"   Final: {self.final_dir}")

    def save_checkpoint(self, checkpoint, epoch, is_best, save_epoch_ckpt=False, epoch_interval=5, keep_n=3):
        def _atomic(obj, path: Path) -> bool:
            tmp = path.with_suffix(f".{self.pid}.tmp")
            try:
                torch.save(obj, tmp)
                tmp.replace(path)
                return True
            except Exception as e:
                try:
                    if tmp.exists():
                        tmp.unlink()
                except Exception:
                    pass
                print(f"⚠️ Save failed ({path.name}): {repr(e)}")
                return False

        _atomic(checkpoint, self.temp_dir / "last_checkpoint.pth")
        if is_best:
            _atomic(checkpoint, self.temp_dir / "best_model.pth")
        if save_epoch_ckpt and (not is_best) and (epoch % max(1, epoch_interval) == 0):
            if _atomic(checkpoint, self.temp_dir / f"checkpoint_epoch_{epoch}.pth"):
                self._cleanup_old_epochs(keep_n)

    def _cleanup_old_epochs(self, keep_n: int):
        ckpts = sorted(
            self.temp_dir.glob("checkpoint_epoch_*.pth"),
            key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.name).group(1))
            if re.search(r"checkpoint_epoch_(\d+)", p.name) else 0
        )
        for old in ckpts[:-keep_n]:
            try:
                old.unlink()
            except Exception:
                pass

    def _save_confusion_matrix_png(self, cm, classes, out_path):
        try:
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(8, 8))
            im = ax.imshow(cm, interpolation="nearest")
            ax.figure.colorbar(im, ax=ax)
            ax.set(
                xticks=np.arange(len(classes)),
                yticks=np.arange(len(classes)),
                xticklabels=classes,
                yticklabels=classes,
                ylabel="True label",
                xlabel="Predicted label",
                title="Confusion Matrix",
            )
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
            thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    ax.text(
                        j, i, format(cm[i, j], "d"),
                        ha="center", va="center",
                        color="white" if cm[i, j] > thresh else "black"
                    )
            fig.tight_layout()
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)
        except Exception as e:
            print(f"⚠️ Could not save confusion_matrix.png: {e}")

    def copy_final_artifacts(self, best_ckpt_path, config, train_log, test_metrics, class_to_idx, classes, logs_dir):
        print("\n" + "=" * 80)
        print("COPYING FINAL ARTIFACTS")
        print("=" * 80)

        if not best_ckpt_path.exists():
            raise FileNotFoundError(f"best_model.pth not found: {best_ckpt_path}")

        model_dir   = self.final_dir / "model"
        config_dir  = self.final_dir / "config"
        logs_dst    = self.final_dir / "logs"
        metrics_dir = self.final_dir / "metrics"
        raw_dir     = self.final_dir / "raw_outputs"
        fig_dir     = self.final_dir / "figures"

        shutil.copy2(best_ckpt_path, model_dir / "best_model.pth")
        best_ckpt = torch.load(best_ckpt_path, map_location="cpu")
        torch.save(best_ckpt["model"], model_dir / "model_weights_only.pth")

        if "ema_shadow" in best_ckpt and isinstance(best_ckpt["ema_shadow"], dict):
            ema_cpu = {k: v.detach().cpu() for k, v in best_ckpt["ema_shadow"].items()}
            merged  = dict(best_ckpt["model"])
            for k, v in ema_cpu.items():
                if k in merged:
                    merged[k] = v
            torch.save(merged, model_dir / "ema_model_weights_only.pth")

        with open(config_dir / "config.json", "w") as f:
            json.dump(config.__dict__, f, indent=2)
        with open(config_dir / "class_to_idx.json", "w") as f:
            json.dump(class_to_idx, f, indent=2)
        with open(config_dir / "classes.json", "w") as f:
            json.dump(classes, f, indent=2)

        if train_log:
            with open(logs_dst / "train_log.csv", "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=train_log[0].keys())
                writer.writeheader()
                writer.writerows(train_log)

        gate_src = Path(logs_dir) / "gate_stats_log.csv"
        gate_dst = logs_dst / "gate_stats_log.csv"
        if gate_src.exists() and gate_src.resolve() != gate_dst.resolve():
            shutil.copy2(gate_src, gate_dst)

        results = {}
        for k, v in test_metrics.items():
            if not isinstance(v, (np.ndarray, list, dict)):
                sv = _json_safe_scalar(v)
                results[k] = "N/A" if (k == "macro_auc" and sv == -1.0) else sv

        with open(metrics_dir / "test_results.json", "w") as f:
            json.dump(results, f, indent=2)

        if "targets" in test_metrics and "preds" in test_metrics:
            cm = confusion_matrix(test_metrics["targets"], test_metrics["preds"])
            np.save(metrics_dir / "confusion_matrix.npy", cm)
            rep = classification_report(
                test_metrics["targets"], test_metrics["preds"],
                target_names=classes, digits=4
            )
            with open(metrics_dir / "classification_report.txt", "w") as f:
                f.write(rep)

            per_class_f1        = f1_score(test_metrics["targets"], test_metrics["preds"], average=None, labels=list(range(len(classes))))
            per_class_precision = precision_score(test_metrics["targets"], test_metrics["preds"], average=None, labels=list(range(len(classes))), zero_division=0)
            per_class_recall    = recall_score(test_metrics["targets"], test_metrics["preds"], average=None, labels=list(range(len(classes))), zero_division=0)
            per_class = {}
            for i, cn in enumerate(classes):
                per_class[cn] = {
                    "f1":        float(per_class_f1[i]),
                    "precision": float(per_class_precision[i]),
                    "recall":    float(per_class_recall[i]),
                }
            with open(metrics_dir / "per_class_metrics.json", "w") as f:
                json.dump(per_class, f, indent=2)

            if config.save_cm_png:
                self._save_confusion_matrix_png(cm, classes, fig_dir / "confusion_matrix.png")

        for key, fname in [
            ("preds",   "test_predictions.npy"),
            ("targets", "test_targets.npy"),
            ("probs",   "test_probabilities.npy"),
        ]:
            if key in test_metrics:
                np.save(raw_dir / fname, test_metrics[key])

        if "gate_stats" in test_metrics:
            with open(raw_dir / "gate_stats_test.json", "w") as f:
                json.dump(test_metrics["gate_stats"], f, indent=2)

        print(f"✅ Final bundle ready at: {self.final_dir}")


# =============================================================================
# TRAINER
# =============================================================================

class Trainer:
    def __init__(self, cfg: Config):
        cfg.resolve_backbone()
        cfg.validate()
        self.cfg = cfg
        set_seed(cfg.seed, deterministic=cfg.deterministic)

        if cfg.deterministic and cfg.num_workers > 0:
            print("⚠️ num_workers > 0 with deterministic=True may introduce small non-determinism.")

        self.run_root = Path(cfg.run_dir)
        self.run_root.mkdir(parents=True, exist_ok=True)
        self.exp_dir = self.run_root / cfg.experiment_name
        self.exp_dir.mkdir(parents=True, exist_ok=True)
        self.logs_dir = self.exp_dir / "logs"
        self.logs_dir.mkdir(exist_ok=True)
        self.config_dir = self.exp_dir / "config"
        self.config_dir.mkdir(exist_ok=True)

        self.ckpt_manager = CheckpointManager(
            temp_root=Path(cfg.ckpt_temp_dir),
            final_root=self.run_root,
            experiment_name=cfg.experiment_name,
        )

        self.model = RGBHSVModel(cfg).to(cfg.device)
        self.params_m   = float(count_params_m(self.model))
        self.gflops_out = -1.0

        print(f"   Params: {self.params_m:.2f}M")

        # P100 = single GPU; DataParallel with 1 device = pure overhead.
        # use_data_parallel defaults to False; only activate if >1 GPU found.
        self.n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
        if self.n_gpus > 1 and cfg.use_data_parallel:
            print(f"\n🚀 DataParallel: {self.n_gpus} GPUs")
            self.model = nn.DataParallel(self.model)
        else:
            if torch.cuda.is_available():
                mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
                print(f"\n💻 Single GPU: {torch.cuda.get_device_name(0)}  ({mem:.1f} GB)")
            self.n_gpus = max(self.n_gpus, 1)

        base_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        self.ema = EMA(base_model, cfg.ema_decay) if cfg.use_ema else None

        self.train_loader, self.val_loader, self.test_loader = self._build_loaders()

        self.class_weights = None
        if cfg.use_class_weights:
            class_counts = Counter(self.train_loader.dataset.targets)
            weights = [1.0 / max(int(class_counts.get(i, 0)), 1) for i in range(cfg.num_classes)]
            mean_w  = sum(weights) / max(len(weights), 1)
            weights = [w / max(mean_w, 1e-12) for w in weights]
            self.class_weights = torch.tensor(weights, dtype=torch.float32, device=cfg.device)
            print(f"\n⚖️  Class weights: {[f'{w:.3f}' for w in weights]}")

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=cfg.lr,
            weight_decay=cfg.weight_decay,
            betas=(0.9, 0.999),
        )

        batches_per_epoch  = len(self.train_loader)
        accum              = int(cfg.gradient_accumulation_steps)
        updates_per_epoch  = int(math.ceil(batches_per_epoch / accum))
        total_updates      = int(cfg.epochs * updates_per_epoch)
        warmup_updates     = min(int(cfg.warmup_epochs * updates_per_epoch), max(0, total_updates - 1))

        print("\n📅 Scheduler:")
        print(f"   batches/epoch={batches_per_epoch}")
        print(f"   updates/epoch={updates_per_epoch}, warmup={warmup_updates}, total={total_updates}")
        print(f"   effective_batch_size={cfg.batch_size * cfg.gradient_accumulation_steps}")

        if warmup_updates > 0:
            self.scheduler = SequentialLR(
                self.optimizer,
                schedulers=[
                    LinearLR(
                        self.optimizer,
                        start_factor=cfg.warmup_lr_init / cfg.lr,
                        end_factor=1.0,
                        total_iters=warmup_updates,
                    ),
                    CosineAnnealingLR(
                        self.optimizer,
                        T_max=max(1, total_updates - warmup_updates),
                        eta_min=cfg.min_lr,
                    ),
                ],
                milestones=[warmup_updates],
            )
        else:
            self.scheduler = CosineAnnealingLR(
                self.optimizer,
                T_max=max(1, total_updates),
                eta_min=cfg.min_lr,
            )

        self.scaler      = make_grad_scaler(cfg.use_amp)
        self.best_val_f1 = -1.0
        self.bad_epochs  = 0
        self.start_epoch = 1
        self.train_log: List[Dict] = []

        self._save_config()
        self._save_env()
        self._validate_dataset()

    def _save_config(self):
        with open(self.config_dir / "config.json", "w") as f:
            json.dump(self.cfg.__dict__, f, indent=2)

    def _save_env(self):
        info = {
            "torch":    torch.__version__,
            "timm":     timm.__version__,
            "cuda":     torch.version.cuda if torch.cuda.is_available() else None,
            "gpu0":     torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "num_gpus": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        }
        with open(self.config_dir / "env.json", "w") as f:
            json.dump(info, f, indent=2)

    def _validate_dataset(self):
        print("\n🔍 Dataset split sizes:")
        for name, ldr in [("Train", self.train_loader), ("Val", self.val_loader), ("Test", self.test_loader)]:
            print(f"   {name}: {len(ldr.dataset)}")

    def _build_loaders(self):
        root = Path(self.cfg.data_root)
        for s in ["train", "val", "test"]:
            if not (root / s).exists():
                raise FileNotFoundError(f"Missing split: {root / s}")

        hue_jitter = 0.0 if self.cfg.use_hsv_branch else self.cfg.hue_jitter

        train_tf_list = [
            T.RandomResizedCrop(
                self.cfg.input_size,
                scale=(self.cfg.rrc_scale_min, 1.0),
                interpolation=InterpolationMode.BICUBIC,
            ),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(
                brightness=self.cfg.color_jitter,
                contrast=self.cfg.color_jitter,
                saturation=self.cfg.color_jitter,
                hue=hue_jitter,
            ),
        ]
        if self.cfg.use_gaussian_blur:
            train_tf_list.append(
                T.RandomApply([T.GaussianBlur(kernel_size=3)], p=self.cfg.gaussian_blur_prob)
            )
        train_tf_list += [
            T.ToTensor(),
            T.Normalize(mean=self.cfg.img_mean, std=self.cfg.img_std),
        ]

        eval_tf = T.Compose([
            T.Resize(int(self.cfg.input_size * 1.14), interpolation=InterpolationMode.BICUBIC),
            T.CenterCrop(self.cfg.input_size),
            T.ToTensor(),
            T.Normalize(mean=self.cfg.img_mean, std=self.cfg.img_std),
        ])

        train_ds = ImageFolder(root / "train", transform=T.Compose(train_tf_list))
        val_ds   = ImageFolder(root / "val",   transform=eval_tf)
        test_ds  = ImageFolder(root / "test",  transform=eval_tf)

        if len(train_ds.classes) != self.cfg.num_classes:
            raise ValueError(
                f"Expected {self.cfg.num_classes} classes, "
                f"found {len(train_ds.classes)}: {train_ds.classes}"
            )

        print(f"\n📦 Data: Train={len(train_ds)} | Val={len(val_ds)} | Test={len(test_ds)}")

        use_persistent = self.cfg.num_workers > 0
        pin_mem        = torch.cuda.is_available()   # async H→D copies help even with num_workers=0

        common_kwargs = dict(
            num_workers=self.cfg.num_workers,
            pin_memory=pin_mem,
            persistent_workers=use_persistent,
        )
        if self.cfg.num_workers > 0:
            common_kwargs["prefetch_factor"] = 4

        train_loader = DataLoader(
            train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            drop_last=False,
            worker_init_fn=worker_init_fn,
            generator=torch.Generator().manual_seed(self.cfg.seed),
            **common_kwargs,
        )
        # Eval loaders: 2x batch safe on P100 (no grad, ~half the VRAM)
        val_loader = DataLoader(
            val_ds,
            batch_size=self.cfg.batch_size * 2,
            shuffle=False,
            **common_kwargs,
        )
        test_loader = DataLoader(
            test_ds,
            batch_size=self.cfg.batch_size * 2,
            shuffle=False,
            **common_kwargs,
        )
        return train_loader, val_loader, test_loader

    def _gate_alpha_for_epoch(self, epoch: int) -> float:
        if not self.cfg.use_hsv_branch:
            return 1.0
        w = max(1, int(self.cfg.gate_warmup_epochs))
        return float(min(1.0, epoch / w))

    def _append_gate_csv(self, epoch, gate_stats, gate_global_mean):
        gate_csv = self.logs_dir / "gate_stats_log.csv"
        row = {"epoch": epoch, "gate_global_mean": gate_global_mean}
        for cname, stats in gate_stats.items():
            row[f"{cname}_mean"] = stats["mean"]
            row[f"{cname}_std"]  = stats["std"]
            row[f"{cname}_n"]    = stats.get("n", 0)
        write_header = not gate_csv.exists()
        with open(gate_csv, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            if write_header:
                writer.writeheader()
            writer.writerow(row)

    def _compute_gate_stats(self, gates_1d, targets, class_names, class_to_idx):
        out = {}
        for cname in class_names:
            ci  = class_to_idx[cname]
            idx = targets == ci
            if idx.sum() == 0:
                out[cname] = {"mean": 0.0, "std": 0.0, "n": 0}
            else:
                vals = gates_1d[idx]
                out[cname] = {"mean": float(vals.mean()), "std": float(vals.std()), "n": int(idx.sum())}
        return out

    def train_one_epoch(self, epoch: int) -> Dict:
        self.model.train()
        total_loss, correct, n = 0.0, 0, 0
        accum      = int(self.cfg.gradient_accumulation_steps)
        gate_alpha = self._gate_alpha_for_epoch(epoch)
        self.optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(self.train_loader, desc=f"Train {epoch}/{self.cfg.epochs}", leave=False)

        for i, (x, y) in enumerate(pbar, start=1):
            x = x.to(self.cfg.device, non_blocking=True)
            y = y.to(self.cfg.device, non_blocking=True)

            with get_autocast_ctx(self.cfg.use_amp):
                logits = self.model(x, gate_alpha=gate_alpha)
                loss = F.cross_entropy(
                    logits, y,
                    weight=self.class_weights,
                    label_smoothing=self.cfg.label_smoothing,
                )

            correct += (logits.argmax(dim=1) == y).sum().item()
            self.scaler.scale(loss / accum).backward()

            do_step = (i % accum == 0) or (i == len(self.train_loader))
            if do_step:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                if self.ema is not None:
                    self.ema.update()
                self.optimizer.zero_grad(set_to_none=True)

            bs = x.size(0)
            total_loss += loss.item() * bs
            n          += bs

            if i % self.cfg.log_interval == 0:
                lr = self.optimizer.param_groups[0]["lr"]
                pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        return {"loss": total_loss / max(1, n), "acc1": correct / max(1, n)}

    @torch.no_grad()
    def evaluate(self, loader, use_ema=False, return_arrays=False, compute_auc=True) -> Dict:
        if use_ema and self.ema is not None:
            self.ema.apply_shadow()
        self.model.eval()

        all_preds, all_targets = [], []
        all_probs  = [] if compute_auc else None
        all_gates  = []
        total_loss, correct, n = 0.0, 0, 0

        class_names  = self.train_loader.dataset.classes
        class_to_idx = self.train_loader.dataset.class_to_idx

        for x, y in tqdm(loader, desc="Eval", leave=False):
            x = x.to(self.cfg.device, non_blocking=True)
            y = y.to(self.cfg.device, non_blocking=True)

            with get_autocast_ctx(self.cfg.use_amp):
                out = self.model(x, return_gate=True, gate_alpha=1.0)
                if isinstance(out, (tuple, list)) and len(out) == 2:
                    logits, gate = out
                else:
                    logits, gate = out, None
                loss = F.cross_entropy(logits, y, weight=self.class_weights)

            probs  = F.softmax(logits.float(), dim=1)
            preds  = probs.argmax(dim=1)
            correct    += (preds == y).sum().item()
            bs          = x.size(0)
            total_loss += loss.item() * bs
            n          += bs

            all_preds.extend(preds.cpu().numpy().tolist())
            all_targets.extend(y.cpu().numpy().tolist())
            if compute_auc and all_probs is not None:
                all_probs.append(probs.cpu().numpy())
            if gate is not None:
                all_gates.append(gate.detach().float().cpu().numpy())

        if use_ema and self.ema is not None:
            self.ema.restore()

        preds_np   = np.array(all_preds)
        targets_np = np.array(all_targets)

        macro_auc = -1.0
        probs_np  = None
        if compute_auc and all_probs:
            probs_np = np.concatenate(all_probs, axis=0).astype(np.float64)
            try:
                macro_auc = roc_auc_score(
                    targets_np, probs_np,
                    multi_class="ovr", average="macro",
                    labels=np.arange(len(class_names)),
                )
            except Exception as e:
                print(f"⚠️ AUC failed: {e}")

        out_dict = {
            "loss":            float(total_loss / max(1, n)),
            "acc1":            float(correct / max(1, n)),
            "macro_f1":        float(f1_score(targets_np, preds_np, average="macro")),
            "micro_f1":        float(f1_score(targets_np, preds_np, average="micro")),
            "weighted_f1":     float(f1_score(targets_np, preds_np, average="weighted")),
            "macro_precision": float(precision_score(targets_np, preds_np, average="macro", zero_division=0)),
            "macro_recall":    float(recall_score(targets_np, preds_np, average="macro", zero_division=0)),
            "macro_auc":       float(macro_auc),
        }

        if return_arrays:
            out_dict["preds"]   = preds_np
            out_dict["targets"] = targets_np
            if probs_np is not None:
                out_dict["probs"] = probs_np

        if self.cfg.use_hsv_branch and len(all_gates) > 0:
            gates_np = np.concatenate(all_gates, axis=0).reshape(-1)
            out_dict["gate_stats"]       = self._compute_gate_stats(gates_np, targets_np, class_names, class_to_idx)
            out_dict["gate_global_mean"] = float(gates_np.mean())
            if return_arrays:
                out_dict["gates"] = gates_np

        return out_dict

    @torch.no_grad()
    def benchmark_inference(self, loader, warmup_batches=5):
        self.model.eval()
        if len(loader) <= warmup_batches:
            return -1.0, -1.0
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time, n_images = None, 0
        for bi, (x, _) in enumerate(loader):
            x = x.to(self.cfg.device, non_blocking=True)
            with get_autocast_ctx(self.cfg.use_amp):
                _ = self.model(x, gate_alpha=1.0)
            if bi == warmup_batches:
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                start_time = time.perf_counter()
                n_images   = 0
            if start_time is not None:
                n_images += x.size(0)

        if start_time is None or n_images == 0:
            return -1.0, -1.0
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start_time
        return float(elapsed), float(n_images / max(elapsed, 1e-9))

    def save_checkpoint(self, epoch, val_metrics, is_best):
        base = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        ckpt = {
            "epoch":       int(epoch),
            "model":       base.state_dict(),
            "optimizer":   self.optimizer.state_dict(),
            "scheduler":   self.scheduler.state_dict(),
            "scaler":      self.scaler.state_dict(),
            "config":      self.cfg.__dict__,
            "best_val_f1": float(self.best_val_f1),
            "bad_epochs":  int(self.bad_epochs),
        }
        if self.ema is not None and is_best:
            # Serialise shadow to CPU: keeps .pth portable and small.
            # Shadow lives on GPU during training (P100), so .cpu() is needed here.
            ckpt["ema_shadow"] = {
                k: v.detach().cpu().clone() for k, v in self.ema.shadow.items()
            }

        self.ckpt_manager.save_checkpoint(
            ckpt, epoch, is_best,
            save_epoch_ckpt=self.cfg.save_epoch_checkpoints,
            epoch_interval=self.cfg.save_epoch_every,
            keep_n=self.cfg.keep_last_n_checkpoints,
        )

    def save_train_log(self):
        if not self.train_log:
            return
        with open(self.logs_dir / "train_log.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.train_log[0].keys())
            writer.writeheader()
            writer.writerows(self.train_log)

    def resume_from(self, ckpt_path: Path):
        print(f"\n📂 Resuming from: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location="cpu")
        base = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        load_state_dict_robust(base, ckpt["model"])

        for name, obj in [
            ("optimizer", self.optimizer),
            ("scheduler",  self.scheduler),
            ("scaler",     self.scaler),
        ]:
            if name in ckpt:
                try:
                    obj.load_state_dict(ckpt[name])
                except Exception as e:
                    print(f"⚠️ {name} load failed: {e}")

        if self.ema is not None and "ema_shadow" in ckpt:
            try:
                # Move shadow to GPU (P100: shadow lives on device during training)
                self.ema.shadow = {
                    k: v.to(self.cfg.device, non_blocking=True)
                    for k, v in ckpt["ema_shadow"].items()
                }
                print("✅ EMA state loaded")
            except Exception as e:
                print(f"⚠️ EMA load failed: {e}")

        self.best_val_f1 = float(ckpt.get("best_val_f1", self.best_val_f1))
        self.bad_epochs  = int(ckpt.get("bad_epochs",   self.bad_epochs))
        self.start_epoch = max(1, int(ckpt.get("epoch", 0)) + 1)
        print(f"✅ Resumed: next epoch {self.start_epoch}, best F1 {self.best_val_f1:.4f}")

    def fit(self):
        print("\n" + "=" * 80)
        print("START TRAINING")
        print("=" * 80)
        print(f"Experiment:      {self.cfg.experiment_name}")
        print(f"Backbone:        {BACKBONE_KEY} ({self.cfg.timm_id})")
        print(f"feat_dim:        {self.cfg.feat_dim}")
        print(f"HSV branch:      {self.cfg.use_hsv_branch} | EMA: {self.cfg.use_ema}")
        print(f"AMP:             {self.cfg.use_amp and torch.cuda.is_available()}")
        print(f"Batch size:      {self.cfg.batch_size}")
        print(f"Grad accum:      {self.cfg.gradient_accumulation_steps}")
        print(f"Effective batch: {self.cfg.batch_size * self.cfg.gradient_accumulation_steps}")
        print(f"DataParallel:    {isinstance(self.model, nn.DataParallel)}")
        print("=" * 80)

        effective_val_ema_used = False

        for epoch in range(self.start_epoch, self.cfg.epochs + 1):
            train_m = self.train_one_epoch(epoch)

            use_ema_val = (
                self.cfg.use_ema and
                (self.ema is not None) and
                (epoch % self.cfg.ema_eval_every == 0 or epoch == self.cfg.epochs)
            )

            val_m = self.evaluate(
                self.val_loader,
                use_ema=use_ema_val,
                return_arrays=False,
                compute_auc=self.cfg.compute_val_auc,
            )

            lr  = float(self.optimizer.param_groups[0]["lr"])
            tag = " (EMA)" if use_ema_val else ""
            print(f"\nEpoch {epoch}/{self.cfg.epochs} | LR: {lr:.2e}")
            print(f"  Train     Loss: {train_m['loss']:.4f}  Acc@1: {train_m['acc1']:.4f}")
            print(f"  Val{tag:6s}  Loss: {val_m['loss']:.4f}  Acc@1: {val_m['acc1']:.4f}  Macro-F1: {val_m['macro_f1']:.4f}")

            if self.cfg.use_hsv_branch and "gate_stats" in val_m:
                self._append_gate_csv(epoch, val_m["gate_stats"], val_m.get("gate_global_mean", 0.0))

            self.train_log.append({
                "epoch":                int(epoch),
                "lr":                   float(lr),
                "train_loss":           float(train_m["loss"]),
                "train_acc1":           float(train_m["acc1"]),
                "val_loss":             float(val_m["loss"]),
                "val_acc1":             float(val_m["acc1"]),
                "val_macro_f1":         float(val_m["macro_f1"]),
                "val_micro_f1":         float(val_m["micro_f1"]),
                "val_weighted_f1":      float(val_m["weighted_f1"]),
                "val_macro_precision":  float(val_m["macro_precision"]),
                "val_macro_recall":     float(val_m["macro_recall"]),
                "val_macro_auc":        float(val_m["macro_auc"]),
                "val_gate_global_mean": float(val_m.get("gate_global_mean", 0.0)),
            })
            self.save_train_log()

            improved = float(val_m["macro_f1"]) > self.best_val_f1
            if improved:
                effective_val_ema_used = use_ema_val
                self.best_val_f1 = float(val_m["macro_f1"])
                self.bad_epochs  = 0
                self.save_checkpoint(epoch, val_m, is_best=True)
                print(f"  🌟 New best Macro-F1: {self.best_val_f1:.4f}")
            else:
                self.bad_epochs += 1
                self.save_checkpoint(epoch, val_m, is_best=False)
                print(f"  No improvement ({self.bad_epochs}/{self.cfg.early_stopping_patience})")

            if self.bad_epochs >= self.cfg.early_stopping_patience:
                print("\n⏹️ Early stopping triggered.")
                break

        print("\n" + "=" * 80)
        print("FINAL TEST EVALUATION")
        print("=" * 80)

        best_path = self.ckpt_manager.temp_dir / "best_model.pth"
        if not best_path.exists():
            raise FileNotFoundError(f"No best_model.pth in {self.ckpt_manager.temp_dir}")

        best_ckpt  = torch.load(best_path, map_location="cpu")
        base_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model

        if self.cfg.use_ema and "ema_shadow" in best_ckpt:
            print(f"📊 Loading EMA weights (epoch {best_ckpt.get('epoch', '?')})")
            merged = dict(best_ckpt["model"])
            for k, v in best_ckpt["ema_shadow"].items():
                if k in merged:
                    merged[k] = v.to(self.cfg.device, non_blocking=True)
            load_state_dict_robust(base_model, merged)
            used_ema_weights = True
        else:
            if self.cfg.use_ema:
                warnings.warn("EMA enabled but ema_shadow missing. Using raw weights.")
            print(f"📊 Loading raw weights (epoch {best_ckpt.get('epoch', '?')})")
            load_state_dict_robust(base_model, best_ckpt["model"])
            used_ema_weights = False

        test_m = self.evaluate(self.test_loader, use_ema=False, return_arrays=True, compute_auc=True)
        test_m["used_ema_weights"] = used_ema_weights

        elapsed, throughput = self.benchmark_inference(self.test_loader)
        test_m.update({
            "inference_seconds": float(elapsed),
            "throughput_img_s":  float(throughput),
            "params_m":          float(self.params_m),
            "gflops":            float(self.gflops_out),
            "backbone_key":      BACKBONE_KEY,
            "timm_id":           self.cfg.timm_id,
            "use_hsv_branch":    self.cfg.use_hsv_branch,
            "model_selection": {
                "criterion":          "macro_f1",
                "use_ema":            bool(self.cfg.use_ema),
                "effective_ema_val":  bool(effective_val_ema_used),
                "effective_ema_test": bool(used_ema_weights),
                "best_epoch":         int(best_ckpt.get("epoch", -1)),
                "best_val_f1":        float(self.best_val_f1),
            },
        })

        print(f"\n📊 Test Results:")
        print(f"   Acc@1:     {test_m['acc1']:.4f}")
        print(f"   Macro-F1:  {test_m['macro_f1']:.4f}")
        auc_str = f"{test_m['macro_auc']:.4f}" if test_m["macro_auc"] >= 0 else "N/A"
        print(f"   Macro-AUC: {auc_str}")
        print(f"\n⚙️  Params: {test_m['params_m']:.2f}M")
        if elapsed >= 0:
            print(f"⏱️  Throughput: {throughput:.1f} img/s")

        self.ckpt_manager.copy_final_artifacts(
            best_ckpt_path=best_path,
            config=self.cfg,
            train_log=self.train_log,
            test_metrics=test_m,
            class_to_idx=self.train_loader.dataset.class_to_idx,
            classes=self.train_loader.dataset.classes,
            logs_dir=self.logs_dir,
        )
        return test_m


# =============================================================================
# CLI
# =============================================================================

def parse_args():
    p = argparse.ArgumentParser(
        description="ConvNeXt-V2-T | P100-optimised training script",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )

    p.add_argument("--pretrained",      action="store_true")
    p.add_argument("--drop_path_rate",  type=float, default=0.2)

    p.add_argument("--exp_name", type=str, required=True)
    p.add_argument("--run_dir",  type=str, default="/kaggle/working/experiments_ablation")
    p.add_argument(
        "--data_root", type=str,
        default="/kaggle/input/datasets/saifullahsharafatfb/tea-leaf701515/tea_leaf_processed_dataset/tea_leaf_processed_dataset"
    )

    # P100 defaults
    p.add_argument("--batch_size",                  type=int,   default=64)
    p.add_argument("--num_workers",                 type=int,   default=4)
    p.add_argument("--epochs",                      type=int,   default=80)
    p.add_argument("--lr",                          type=float, default=5e-4)
    p.add_argument("--seed",                        type=int,   default=42)
    p.add_argument("--gradient_accumulation_steps", type=int,   default=1)
    p.add_argument("--ema_eval_every",              type=int,   default=5)

    p.add_argument("--use_ema",           action="store_true")
    p.add_argument("--use_class_weights", action="store_true")
    p.add_argument("--compute_val_auc",   action="store_true")
    p.add_argument("--deterministic",     action="store_true",
                   help="Enable full determinism (slower; off by default on P100).")

    p.add_argument("--use_hsv",            action="store_true")
    p.add_argument("--hsv_raw",            action="store_true")
    p.add_argument("--gate_vector",        action="store_true")
    p.add_argument("--gate_warmup_epochs", type=int, default=5)

    p.add_argument("--ckpt_temp_dir",           type=str, default="/kaggle/working/temp")
    p.add_argument("--resume",                  type=str, default=None)
    p.add_argument("--auto_resume",             action="store_true")
    p.add_argument("--save_epoch_checkpoints",  action="store_true")
    p.add_argument("--save_epoch_every",        type=int, default=5)
    p.add_argument("--keep_last_n_checkpoints", type=int, default=3)
    p.add_argument("--enable_dp",   action="store_true",
                   help="Force DataParallel (only useful if >1 GPU is somehow available).")
    p.add_argument("--no_cm_png",   action="store_true")

    return p.parse_args()


def main():
    args = parse_args()

    cfg = Config(
        pretrained=args.pretrained,
        drop_path_rate=args.drop_path_rate,
        data_root=args.data_root,
        batch_size=args.batch_size,
        epochs=args.epochs,
        lr=args.lr,
        num_workers=args.num_workers,
        seed=args.seed,
        deterministic=args.deterministic,
        use_ema=args.use_ema,
        ema_eval_every=args.ema_eval_every,
        use_class_weights=args.use_class_weights,
        experiment_name=args.exp_name,
        use_hsv_branch=args.use_hsv,
        gate_vector=args.gate_vector,
        hsv_use_sincos=(not args.hsv_raw),
        gate_warmup_epochs=int(args.gate_warmup_epochs),
        ckpt_temp_dir=args.ckpt_temp_dir,
        save_epoch_checkpoints=args.save_epoch_checkpoints,
        save_epoch_every=args.save_epoch_every,
        keep_last_n_checkpoints=args.keep_last_n_checkpoints,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        compute_val_auc=args.compute_val_auc,
        run_dir=args.run_dir,
        use_data_parallel=args.enable_dp,   # False by default on P100
        save_cm_png=(not args.no_cm_png),
    )

    print("=" * 80)
    print("CONVNEXTV2-T  |  P100-OPTIMISED TRAINING SCRIPT")
    print("=" * 80)
    print(f"CUDA:     {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({mem:.1f} GB)")
    print(f"PyTorch:  {torch.__version__}")
    print(f"timm:     {timm.__version__}")
    print("=" * 80)

    trainer = Trainer(cfg)

    if args.resume is not None:
        trainer.resume_from(Path(args.resume))
    elif args.auto_resume:
        last = trainer.ckpt_manager.temp_dir / "last_checkpoint.pth"
        if last.exists():
            trainer.resume_from(last)
        else:
            print("ℹ️  No last_checkpoint.pth found. Starting fresh.")

    trainer.fit()


if __name__ == "__main__":
    main()

Overwriting convtea.py


In [3]:
!python convtea.py \
  --exp_name convnextv2t_rgb_ema_seed42_p100 \
  --pretrained \
  --use_ema \
  --batch_size 64 \
  --gradient_accumulation_steps 1 \
  --num_workers 4 \
  --seed 42

CONVNEXTV2-T  |  P100-OPTIMISED TRAINING SCRIPT
CUDA:     True
  GPU 0: Tesla P100-PCIE-16GB  (15.9 GB)
PyTorch:  2.11.0+cu126
timm:     1.0.25

🔍 Backbone resolved:
   key:      convnextv2_t
   timm_id:  convnextv2_tiny.fcmae_ft_in1k
   feat_dim: 768
   notes:    ConvNeXt-V2-Tiny only

📁 Directories:
   Temp:  /kaggle/working/temp/convnextv2t_rgb_ema_seed42_p100
   Final: /kaggle/working/experiments_ablation/convnextv2t_rgb_ema_seed42_p100
model.safetensors: 100%|█████████████████████| 115M/115M [00:02<00:00, 56.9MB/s]
   Params: 27.87M

💻 Single GPU: Tesla P100-PCIE-16GB  (15.9 GB)

📦 Data: Train=6091 | Val=851 | Test=852

📅 Scheduler:
   batches/epoch=96
   updates/epoch=96, warmup=480, total=7680
   effective_batch_size=64

🔍 Dataset split sizes:
   Train: 6091
   Val: 851
   Test: 852

START TRAINING
Experiment:      convnextv2t_rgb_ema_seed42_p100
Backbone:        convnextv2_t (convnextv2_tiny.fcmae_ft_in1k)
feat_dim:        768
HSV branch:      False | EMA: True
AMP:            

In [5]:
!python convtea.py \
    --exp_name convnextv2t_hsvraw_ema_seed42_p100 \
    --pretrained --use_ema --use_hsv --hsv_raw --batch_size 64 \
    --gradient_accumulation_steps 1 --num_workers 4 --seed 42

CONVNEXTV2-T  |  P100-OPTIMISED TRAINING SCRIPT
CUDA:     True
  GPU 0: Tesla P100-PCIE-16GB  (15.9 GB)
PyTorch:  2.11.0+cu126
timm:     1.0.25

🔍 Backbone resolved:
   key:      convnextv2_t
   timm_id:  convnextv2_tiny.fcmae_ft_in1k
   feat_dim: 768
   notes:    ConvNeXt-V2-Tiny only

📁 Directories:
   Temp:  /kaggle/working/temp/convnextv2t_hsvraw_ema_seed42_p100
   Final: /kaggle/working/experiments_ablation/convnextv2t_hsvraw_ema_seed42_p100
   Params: 28.50M

💻 Single GPU: Tesla P100-PCIE-16GB  (15.9 GB)

📦 Data: Train=6091 | Val=851 | Test=852

📅 Scheduler:
   batches/epoch=96
   updates/epoch=96, warmup=480, total=7680
   effective_batch_size=64

🔍 Dataset split sizes:
   Train: 6091
   Val: 851
   Test: 852

START TRAINING
Experiment:      convnextv2t_hsvraw_ema_seed42_p100
Backbone:        convnextv2_t (convnextv2_tiny.fcmae_ft_in1k)
feat_dim:        768
HSV branch:      True | EMA: True
AMP:             True
Batch size:      64
Grad accum:      1
Effective batch: 64
DataPara

In [6]:
!python convtea.py \
    --exp_name convnextv2t_hsvsincos_scalar_ema_seed42_p100 \
    --pretrained --use_ema --use_hsv --batch_size 64 \
    --gradient_accumulation_steps 1 \
    --num_workers 4 --seed 42

CONVNEXTV2-T  |  P100-OPTIMISED TRAINING SCRIPT
CUDA:     True
  GPU 0: Tesla P100-PCIE-16GB  (15.9 GB)
PyTorch:  2.11.0+cu126
timm:     1.0.25

🔍 Backbone resolved:
   key:      convnextv2_t
   timm_id:  convnextv2_tiny.fcmae_ft_in1k
   feat_dim: 768
   notes:    ConvNeXt-V2-Tiny only

📁 Directories:
   Temp:  /kaggle/working/temp/convnextv2t_hsvsincos_scalar_ema_seed42_p100
   Final: /kaggle/working/experiments_ablation/convnextv2t_hsvsincos_scalar_ema_seed42_p100
   Params: 28.50M

💻 Single GPU: Tesla P100-PCIE-16GB  (15.9 GB)

📦 Data: Train=6091 | Val=851 | Test=852

📅 Scheduler:
   batches/epoch=96
   updates/epoch=96, warmup=480, total=7680
   effective_batch_size=64

🔍 Dataset split sizes:
   Train: 6091
   Val: 851
   Test: 852

START TRAINING
Experiment:      convnextv2t_hsvsincos_scalar_ema_seed42_p100
Backbone:        convnextv2_t (convnextv2_tiny.fcmae_ft_in1k)
feat_dim:        768
HSV branch:      True | EMA: True
AMP:             True
Batch size:      64
Grad accum:      

In [7]:
!python convtea.py \
    --exp_name convnextv2t_hsvsincos_vector_ema_seed42_p100 \
    --pretrained --use_ema --use_hsv --gate_vector --batch_size 64 \
    --gradient_accumulation_steps 1 \
    --num_workers 4 --seed 42

CONVNEXTV2-T  |  P100-OPTIMISED TRAINING SCRIPT
CUDA:     True
  GPU 0: Tesla P100-PCIE-16GB  (15.9 GB)
PyTorch:  2.11.0+cu126
timm:     1.0.25

🔍 Backbone resolved:
   key:      convnextv2_t
   timm_id:  convnextv2_tiny.fcmae_ft_in1k
   feat_dim: 768
   notes:    ConvNeXt-V2-Tiny only

📁 Directories:
   Temp:  /kaggle/working/temp/convnextv2t_hsvsincos_vector_ema_seed42_p100
   Final: /kaggle/working/experiments_ablation/convnextv2t_hsvsincos_vector_ema_seed42_p100
   Params: 28.70M

💻 Single GPU: Tesla P100-PCIE-16GB  (15.9 GB)

📦 Data: Train=6091 | Val=851 | Test=852

📅 Scheduler:
   batches/epoch=96
   updates/epoch=96, warmup=480, total=7680
   effective_batch_size=64

🔍 Dataset split sizes:
   Train: 6091
   Val: 851
   Test: 852

START TRAINING
Experiment:      convnextv2t_hsvsincos_vector_ema_seed42_p100
Backbone:        convnextv2_t (convnextv2_tiny.fcmae_ft_in1k)
feat_dim:        768
HSV branch:      True | EMA: True
AMP:             True
Batch size:      64
Grad accum:      